This notebook plots the results of notebook 03: how the linear probe trained, how each method's setting was chosen, and how well each of the 6 models (2 methods × 3 feature sets) predicts crops on the held-out test blocks, overall, by country and by crop. Three scores are shown: accuracy (share of fields labelled correctly), recall (share of each crop's fields found, averaged over crops, also called mean recall or balanced accuracy) and F1 (balances recall with precision, averaged over crops, also called macro F1).

In [ ]:
import numpy as np
import pandas as pd
import ultraplot as uplt
from pathlib import Path
from sklearn.metrics import f1_score,recall_score

In [ ]:
INTERIMDIR  = Path('../data/interim')
PROCDIR     = Path('../data/processed')
FIGDIR      = Path('../figs')
COLLECTIONS = pd.read_csv(INTERIMDIR/'collections.csv')
NAMES       = dict(zip(COLLECTIONS.collection.str.split('/').str[1],COLLECTIONS.name.str.replace(r'\s*\(.*\)$','',regex=True)))
METHODS     = {'probe':'Linear Probe','knn':'kNN'}
FEATURESETS = {'olmoearth':'OlmoEarth','aef':'AEF','both':'Both'}
COLORS      = {'olmoearth':'#eb6834','aef':'#2a78d6','both':'#1baf7a'}
METRICS     = {'accuracy':'Accuracy','meanrecall':'Recall','macrof1':'F1'}
TOPCROPS    = 25

FIGDIR.mkdir(parents=True,exist_ok=True)

In [ ]:
predictions = pd.read_parquet(INTERIMDIR/'results_predictions.parquet').assign(country=lambda table:table.country.map(NAMES))
losses      = pd.read_csv(PROCDIR/'results_probelosses.csv')
probesweep  = pd.read_csv(PROCDIR/'results_probesweep.csv')
knnsweep    = pd.read_csv(PROCDIR/'results_knnsweep.csv')
models      = [(method,featureset) for method in METHODS for featureset in FEATURESETS]

In [ ]:
def score(group,method,featureset):
    truth,pred = group.crop,group[f'{method}_{featureset}']
    crops      = truth.unique()
    return {'accuracy':(truth==pred).mean(),
            'meanrecall':recall_score(truth,pred,labels=crops,average='macro',zero_division=0),
            'macrof1':f1_score(truth,pred,labels=crops,average='macro',zero_division=0)}

scores = pd.DataFrame([{'country':country,'method':method,'featureset':featureset,**score(group,method,featureset)}
                       for country,group in [('All countries',predictions)]+list(predictions.groupby('country')) for method,featureset in models])
scores.to_csv(PROCDIR/'results_scores.csv',index=False)

## Training and model selection

All four panels use the validation blocks, never the test blocks. (a) Class-weighted cross-entropy loss of the final linear probe for each feature set, on training fields (dashed) and validation fields (solid), by optimiser iteration. Validation loss rising while training loss keeps falling means the probe is overfitting; each fit keeps the weights from its lowest validation loss. (b) Validation loss during the penalty sweep, one line per penalty. (c) Validation F1 for each penalty; the best is used for the final probes. (d) Validation F1 for each number of neighbours k; each feature set uses its own best k.

In [ ]:
fig,axs = uplt.subplots(nrows=2,ncols=2,refwidth=3,refheight=2.2,share=False)
axs.format(abc='(a)',abcloc='l',suptitle='Training and Setting Selection (Validation Blocks)')
for featureset,label in FEATURESETS.items():
    curve = losses.query("stage=='final' and featureset==@featureset")
    axs[0].plot(curve.iterations.to_numpy(),curve.trainloss.to_numpy(),color=COLORS[featureset],linestyle='--')
    axs[0].plot(curve.iterations.to_numpy(),curve.valloss.to_numpy(),color=COLORS[featureset],label=label)
for penalty,curve in losses.query("stage=='sweep'").groupby('penalty'):
    axs[1].plot(curve.iterations.to_numpy(),curve.valloss.to_numpy(),label=f'{penalty:,}')
axs[2].plot(probesweep.penalty.to_numpy(),probesweep.valmacrof1.to_numpy(),color=COLORS['both'],marker='o',label='Both')
for featureset,label in FEATURESETS.items():
    sweep = knnsweep.query('featureset==@featureset')
    axs[3].plot(sweep.k.to_numpy(),sweep.valmacrof1.to_numpy(),color=COLORS[featureset],marker='o',label=label)
axs[0].format(title='Final probes: training (dashed) vs. validation (solid) loss',xlabel='Optimiser iterations',ylabel='Class-weighted cross-entropy loss')
axs[1].format(title='Penalty sweep (Both): validation loss',xlabel='Optimiser iterations',ylabel='Class-weighted cross-entropy loss')
axs[2].format(title='Penalty sweep (Both): validation F1',xlabel='L2 penalty strength',ylabel='Validation F1',xscale='log',xformatter='log')
axs[3].format(title='kNN: validation F1 by number of neighbours',xlabel='Number of neighbours (k)',ylabel='Validation F1')
axs[0].legend(loc='cr',ncols=1,frame=False)
axs[1].legend(loc='ur',ncols=1,frame=False,title='L2 penalty')
axs[3].legend(loc='ur',ncols=1,frame=False)
fig.savefig(FIGDIR/'04_training_and_selection.png',dpi=200)

## Scores on the test blocks

In [ ]:
overall = scores[scores.country=='All countries']

fig,axs = uplt.subplots(ncols=3,refwidth=2.2,refheight=1.2,sharey=True,sharex=False)
for ax,(metric,label) in zip(axs,METRICS.items()):
    for featureset,name in FEATURESETS.items():
        subset = overall[overall.featureset==featureset]
        ax.scatter(subset[metric].to_numpy(),subset.method.map(METHODS).to_numpy(),color=COLORS[featureset],label=name,s=40)
    ax.format(title=label,xlabel='Score',ylabel='',xlim=(0,1),ygrid=True)
fig.legend(axs[0].get_legend_handles_labels()[0][:3],list(FEATURESETS.values()),loc='b',ncols=3,frame=False)
fig.format(suptitle='Test-Block Scores, All Countries Pooled')
fig.savefig(FIGDIR/'04_scores_overall.png',dpi=200)

In [ ]:
bycountry = scores[scores.country!='All countries']
order     = bycountry.query("method=='probe' and featureset=='both'").sort_values('macrof1').country.tolist()

fig,axs = uplt.subplots(nrows=2,ncols=3,refwidth=2.2,refheight=0.22*len(order),sharey=True,sharex=True)
for row,method in enumerate(METHODS):
    for col,metric in enumerate(METRICS):
        for featureset,name in FEATURESETS.items():
            subset = bycountry[(bycountry.method==method)&(bycountry.featureset==featureset)].set_index('country').loc[order]
            axs[row,col].scatter(subset[metric].to_numpy(),subset.index.to_numpy(),color=COLORS[featureset],label=name,s=20)
axs.format(xlim=(0,1),xlabel='Score',ylabel='',ygrid=True,toplabels=list(METRICS.values()),leftlabels=list(METHODS.values()),
           suptitle='Test-Block Scores by Country (Sorted by Linear Probe F1 with Both)')
fig.legend(axs[0].get_legend_handles_labels()[0][:3],list(FEATURESETS.values()),loc='b',ncols=3,frame=False)
fig.savefig(FIGDIR/'04_scores_by_country.png',dpi=200)

In [ ]:
top    = predictions.crop.value_counts().head(TOPCROPS).index[::-1]
recall = pd.DataFrame({(method,featureset):recall_score(predictions.crop,predictions[f'{method}_{featureset}'],labels=top,average=None,zero_division=0)
                       for method,featureset in models},index=top)

fig,axs = uplt.subplots(ncols=2,refwidth=2.5,refheight=0.2*TOPCROPS,sharey=True)
for ax,method in zip(axs,METHODS):
    for featureset,name in FEATURESETS.items():
        ax.scatter(recall[(method,featureset)].to_numpy(),top.to_numpy(),color=COLORS[featureset],label=name,s=20)
    ax.format(title=METHODS[method],xlabel="Recall (share of the crop's test fields identified)",ylabel='',xlim=(0,1),ygrid=True,yticklabelsize=7)
fig.legend(axs[0].get_legend_handles_labels()[0][:3],list(FEATURESETS.values()),loc='b',ncols=3,frame=False)
fig.format(suptitle=f'Test-Block Recall for the {TOPCROPS} Most Common Crops, All Countries Pooled')
fig.savefig(FIGDIR/'04_recall_by_crop.png',dpi=200)